In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold
from sklearn.exceptions import NotFittedError

In [2]:
data = pd.read_csv("./grouped_timeseries_csv.csv")

In [3]:
data.head(100)

,date,week,store_id,store_type,region,weekly_sales,temperature,marketing_spend,competition_stores,is_holiday_week
0,2023-01-01,1,Store_1,Mall,North,1247.63,21.2,67.45,4,0
1,2023-01-01,1,Store_2,Standalone,South,792.86,21.2,45.23,3,0
2,2023-01-01,1,Store_3,Mall,North,1456.78,21.2,123.67,5,0
3,2023-01-01,1,Store_4,Outlet,West,634.12,21.2,34.89,2,0
4,2023-01-08,2,Store_1,Mall,North,1289.34,22.1,78.92,3,0
5,2023-01-08,2,Store_2,Standalone,South,785.67,22.1,52.67,4,0
6,2023-01-08,2,Store_3,Mall,North,1498.45,22.1,134.89,2,0
7,2023-01-08,2,Store_4,Outlet,West,645.78,22.1,43.21,5,0
8,2023-01-15,3,Store_1,Mall,North,1324.78,23.5,89.45,4,0
9,2023-01-15,3,Store_2,Standalone,South,798.23,23.5,61.78,3,0


In [4]:
np.unique(data["store_id"].unique())

array(['Store_1', 'Store_2', 'Store_3', 'Store_4'], dtype=object)

In [5]:
X = data[['week', 'temperature', 'marketing_spend']].values
y = data['weekly_sales'].values
groups = data['store_id'].values

In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.exceptions import NotFittedError


def grouped_time_series_kfold(model, X, y, groups, n_folds=4, n_repeats=10, seed=2000):
    scores = []
    np.random.seed(seed)
    unique_groups = np.unique(groups)
    for _ in range(n_repeats):
        gkf = GroupKFold(n_splits=n_folds)
        shuffled_groups = np.random.permutation(unique_groups)
        for train_group_idx, test_group_idx in gkf.split(X, y, groups=shuffled_groups):
            train_groups = shuffled_groups[train_group_idx]
            test_groups = shuffled_groups[test_group_idx]
            # Find the earliest and latest indices for train and test groups
            train_indices = np.where(np.isin(groups, train_groups))[0]
            test_indices = np.where(np.isin(groups, test_groups))[0]
            train_end = np.min(test_indices)  # Ensure temporal order
            train_mask = np.isin(groups, train_groups) & (
                np.arange(len(groups)) < train_end
            )
            test_mask = np.isin(groups, test_groups)
            model.fit(X[train_mask], y[train_mask])
            score = model.score(X[test_mask], y[test_mask])
            scores.append(score)
    return np.array(scores)

In [15]:
def true_time_series_grouped_kfold(model, X, y, groups, time_col, n_folds=3, n_repeats=5, seed=2000):
    """
    Option B: True time-series grouped K-fold with proper temporal constraints
    
    Parameters:
    -----------
    model : sklearn estimator
        The model to validate
    X : array-like, shape (n_samples, n_features)
        Feature matrix
    y : array-like, shape (n_samples,)
        Target values
    groups : array-like, shape (n_samples,)
        Group labels for each sample
    time_col : array-like, shape (n_samples,)
        Time values for each sample (e.g., week numbers)
    n_folds : int
        Number of folds for GroupKFold
    n_repeats : int
        Number of random repetitions
    seed : int
        Random seed for reproducibility
        
    Returns:
    --------
    scores : array
        Cross-validation scores
    """
    scores = []
    fold_details = []
    np.random.seed(seed)
    
    for repeat in range(n_repeats):
        # Create GroupKFold object
        gkf = GroupKFold(n_splits=n_folds)
        
        # CORRECT usage: pass the full arrays to split
        for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
            
            # Get the actual groups in train and test sets
            train_groups = set(groups[train_idx])
            test_groups = set(groups[test_idx])
            
            # Find the earliest time period in the test set
            test_time_periods = time_col[test_idx]
            earliest_test_time = np.min(test_time_periods)
            
            # Create temporal constraint: 
            # Train only on training groups AND time periods before earliest test time
            train_mask = (
                np.isin(groups, list(train_groups)) &  # Must be from training groups
                (time_col < earliest_test_time)         # Must be before test time period
            )
            
            test_mask = np.isin(groups, list(test_groups))
            
            # Check if we have sufficient data
            train_samples = np.sum(train_mask)
            test_samples = np.sum(test_mask)
            
            if train_samples > 0 and test_samples > 0:
                # Train and evaluate
                model.fit(X[train_mask], y[train_mask])
                score = model.score(X[test_mask], y[test_mask])
                scores.append(score)
                
                # Store details for analysis
                fold_info = {
                    'repeat': repeat + 1,
                    'fold': fold + 1,
                    'train_groups': sorted(list(train_groups)),
                    'test_groups': sorted(list(test_groups)),
                    'earliest_test_time': earliest_test_time,
                    'train_time_range': (time_col[train_mask].min(), time_col[train_mask].max()) if train_samples > 0 else (None, None),
                    'test_time_range': (time_col[test_mask].min(), time_col[test_mask].max()),
                    'train_samples': train_samples,
                    'test_samples': test_samples,
                    'score': score
                }
                fold_details.append(fold_info)
                
                print(f"Repeat {repeat+1}, Fold {fold+1}:")
                print(f"  Train groups: {fold_info['train_groups']}")
                print(f"  Test groups: {fold_info['test_groups']}")
                print(f"  Temporal split: train time < {earliest_test_time}")
                print(f"  Train time range: {fold_info['train_time_range']}")
                print(f"  Test time range: {fold_info['test_time_range']}")
                print(f"  Samples: {train_samples} train, {test_samples} test")
                print(f"  Score: {score:.4f}")
                print()
            else:
                print(f"Repeat {repeat+1}, Fold {fold+1}: SKIPPED (insufficient data)")
                print(f"  Train samples: {train_samples}, Test samples: {test_samples}")
                print()
    
    return np.array(scores), fold_details

In [16]:
data["week"].values

array([ 1,  1,  1,  1,  2,  2,  2,  2,  3,  3,  3,  3,  4,  4,  4,  4,  5,
        5,  5,  5,  6,  6,  6,  6,  7,  7,  7,  7,  8,  8,  8,  8,  9,  9,
        9,  9, 10, 10, 10, 10, 11, 11, 11, 11, 12, 12, 12, 12])

In [17]:
true_time_series_grouped_kfold(LinearRegression(), X, y, groups, time_col=data["week"].values)

Repeat 1, Fold 1: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 24

Repeat 1, Fold 2: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 12

Repeat 1, Fold 3: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 12

Repeat 2, Fold 1: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 24

Repeat 2, Fold 2: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 12

Repeat 2, Fold 3: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 12

Repeat 3, Fold 1: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 24

Repeat 3, Fold 2: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 12

Repeat 3, Fold 3: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 12

Repeat 4, Fold 1: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 24

Repeat 4, Fold 2: SKIPPED (insufficient data)
  Train samples: 0, Test samples: 12

Repeat 4, Fold 3: SKIPPED (insufficient data)
  Train samples: 0, Test sampl

(array([], dtype=float64), [])